# NGRIP event ages and uncertainty inputs

This notebook prepares event ages, uncertainty scales and chronology knots for `ngrip_event_age_uncertainty.py`. Paths are relative to the `NGRIP/` directory.

The source workbook contains 34 Greenland Interstadial (GI) and 35 Greenland Stadial (GS) starts selected from [Rasmussen et al. (2014), Table 2](https://doi.org/10.1016/j.quascirev.2014.09.007). GI starts represent warming and GS starts cooling. Original source labels are retained alongside the catalogue labels so that grouped subevents can be traced to the published table.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

# The long sheet retains one row per selected GI or GS boundary.
raw = pd.read_excel(
    "data/raw/Rasmussen2014_GI_GS_starts_no_subevents_wide.xlsx",
    sheet_name="Long_no_subevents",
)
raw.head()

,event_type,event_number,event_label,age_ab2k_yr,age_ka_b2k,ngrip_depth_m,selection_note,maximum_counting_error_yr,notes_comments,source_event_label,definition_uncertainty_code,definition_uncertainty_text,source_pdf_page,source_printed_page
0,GI,1.0,GI-1,14692,14.692,1604.64,lettered subevents collapsed; oldest onset used,186.0,"3, 5",GI-1e,±4,Explicit ±4 yr; retained as working sigma=4 yr,10,9
1,GS,1.0,GS-1,12896,12.896,1526.52,direct Table 2 start,138.0,"3, 5",GS-1,±4,Explicit ±4 yr; retained as working sigma=4 yr,10,9
2,GI,2.1,GI-2.1,23020,23.020,1786.28,direct Table 2 start,583.0,"8, 13",GI-2.1,a,20 yr (1 sigma),10,9
3,GS,2.1,GS-2.1,22900,22.900,1783.62,lettered subevents collapsed; oldest onset used,573.0,"7, 8",GS-2.1c,a,20 yr (1 sigma),10,9
4,GI,2.2,GI-2.2,23340,23.340,1793.19,direct Table 2 start,596.0,"7, 8, 9",GI-2.2,a,20 yr (1 sigma),10,9


## Event ages and uncertainty scales

Ages are converted from ka b2k (before AD 2000) to ka BP1950 by subtracting 0.05 kyr. Uncertainty values are durations and receive no epoch adjustment.

Transition-definition errors are represented by working Gaussian standard deviations: `a` = 20, `b` = 50, `c` = 200, `d` = 100 and `f` = 30 years. The values for `b` and `f` are the midpoints of the published 40–60 and 20–40 year ranges. Explicit ±4-year entries are assigned σ = 4 years, although the source does not identify these entries as 1σ.

Chronology uncertainty is stored separately: the published maximum counting error (MCE) for layer-counted ages, and a working envelope of ±4.5% of b2k age for the model extension. The uniform 4.5% ratio is a study assumption, not a published pointwise uncertainty curve. These envelopes are interpreted approximately as 2σ in the sampler, rather than as hard bounds.

In [2]:
events = raw[[
    "event_label", "source_event_label", "event_type", "age_ka_b2k",
    "definition_uncertainty_code", "maximum_counting_error_yr",
]].copy()
events["event_type"] = events["event_type"].map({"GI": "warming", "GS": "cooling"})
# Convert the age reference from AD 2000 to AD 1950; leave error durations unchanged.
events["age_ka_bp"] = events["age_ka_b2k"] - 0.05
# Convert published definition codes to working standard deviations in years.
definition_sigma = {"±4": 4, "a": 20, "b": 50, "c": 200, "d": 100, "f": 30}
events["definition_sigma_yr"] = events["definition_uncertainty_code"].map(definition_sigma)
# A reported MCE identifies the layer-counted section; older ages use the model envelope.
counted = events["maximum_counting_error_yr"].notna()
events["chronology_envelope_yr"] = np.where(
    counted, events["maximum_counting_error_yr"], 0.045 * events["age_ka_b2k"] * 1000,
)
events["chronology_source"] = np.where(counted, "MCE", "model_ext")
# Keep source labels for traceability and order boundaries from younger to older.
events = events[[
    "event_label", "source_event_label", "event_type", "age_ka_b2k", "age_ka_bp",
    "definition_uncertainty_code", "definition_sigma_yr",
    "chronology_envelope_yr", "chronology_source",
]].sort_values("age_ka_bp").reset_index(drop=True)
events

,event_label,source_event_label,event_type,age_ka_b2k,age_ka_bp,definition_uncertainty_code,definition_sigma_yr,chronology_envelope_yr,chronology_source
0,GS-1,GS-1,cooling,12.896,12.846,±4,4,138.00,MCE
1,GI-1,GI-1e,warming,14.692,14.642,±4,4,186.00,MCE
2,GS-2.1,GS-2.1c,cooling,22.900,22.850,a,20,573.00,MCE
3,GI-2.1,GI-2.1,warming,23.020,22.970,a,20,583.00,MCE
4,GS-2.2,GS-2.2,cooling,23.220,23.170,a,20,590.00,MCE
...,...,...,...,...,...,...,...,...,...
64,GS-24.2,GS-24.2,cooling,106.900,106.850,f,30,4810.50,model_ext
65,GI-24.2,GI-24.2,warming,108.280,108.230,f,30,4872.60,model_ext
66,GS-25,GS-25,cooling,110.640,110.590,f,30,4978.80,model_ext
67,GI-25,GI-25c,warming,115.370,115.320,f,30,5191.65,model_ext


## Chronology grid

The [annual GICC05 age/MCE dataset](https://doi.org/10.1594/PANGAEA.943193) supplies the layer-counted knot scales. The counted section ends at 60.202 ka b2k with an MCE of 2.611 kyr. The primary knot spacing is 5 kyr; 2.5 and 10 kyr grids support the knot-spacing sensitivity analysis.

The sampler accumulates independent Gaussian variance increments at these knots and linearly interpolates their cumulative age offsets to the events. Consequently, an event's interpolated chronology standard deviation can differ from half its tabulated envelope. Monotonicity of the age mapping and event order are enforced during sampling.

In [4]:
annual_path = Path("data/raw/Rasmussen et al-2022-GICC05_time_scale.txt")
# Skip the archive metadata and locate the tab-separated data header.
header = next(i for i, line in enumerate(annual_path.read_text().splitlines())
              if line.startswith("Age [a] (b2k)\t"))
annual = pd.read_csv(annual_path, sep="\t", skiprows=header)
# Express both age and MCE in kyr; anchor the age-error process at zero.
age = np.r_[0.0, annual["Age [a] (b2k)"].to_numpy() / 1000]
mce = np.r_[0.0, annual["MCE [a]"].to_numpy() / 1000]
assert np.allclose([age[-1], mce[-1]], [60.202, 2.611])

grids = []
for spacing in [2.5, 5.0, 10.0]:
    regular = np.arange(0, 120.0, spacing)
    # Replace nearby regular knots with the exact counted endpoint to avoid a short interval.
    regular = regular[np.abs(regular - 60.202) >= spacing / 2]
    knots = np.unique(np.r_[regular, 60.202, 120.0])
    # Interpolate counted MCE; apply the relative working envelope beyond the join.
    envelope = np.where(knots > 60.202, 0.045 * knots, np.interp(knots, age, mce))
    grids.append(pd.DataFrame({
        "knot_spacing_ka": spacing,
        "knot_age_ka_b2k": knots,
        "chronology_envelope_ka": envelope,
    }))
grid = pd.concat(grids, ignore_index=True)
grid.loc[grid["knot_spacing_ka"].eq(5.0)]

,knot_spacing_ka,knot_age_ka_b2k,chronology_envelope_ka
49,5.0,0.000,0.000
50,5.0,5.000,0.011
51,5.0,10.000,0.083
52,5.0,15.000,0.197
53,5.0,20.000,0.444
54,5.0,25.000,0.690
55,5.0,30.000,0.971
56,5.0,35.000,1.298
57,5.0,40.000,1.574
58,5.0,45.000,1.817


## Validation and output

Checks cover catalogue membership and ordering, complete uncertainty fields, agreement between the 44 event-level MCE values and the annual dataset, and nondecreasing knot envelopes.

Two tables are written to `data/processed/`: `ngrip_warming_cooling_starts.csv` contains event labels, ages, definition standard deviations and chronology envelopes with their source; `ngrip_chronology_grid.csv` contains knot ages and envelopes for each spacing. Event uncertainty columns use years; grid ages and envelopes use kyr. Rerunning the notebook replaces these derived tables. The final count table summarizes warming and cooling starts by chronology source.

In [4]:
# Preserve the selected catalogue and its alternating GI/GS stratigraphic order.
assert events["event_label"].is_unique and len(events) == 69
assert events.groupby("event_type").size().to_dict() == {"cooling": 35, "warming": 34}
assert (events["event_type"] != events["event_type"].shift()).all()
assert np.all(np.diff(events["age_ka_bp"]) > 0)
assert events.notna().all().all()
assert events["definition_sigma_yr"].gt(0).all()
assert events["chronology_envelope_yr"].gt(0).all()
# Cross-check the workbook MCE values against the independent annual table.
counted = events["chronology_source"].eq("MCE")
assert counted.sum() == 44
assert np.allclose(events.loc[counted, "chronology_envelope_yr"] / 1000,
                   np.interp(events.loc[counted, "age_ka_b2k"], age, mce))
assert events.loc[counted, "age_ka_b2k"].max() < 60.202
assert events.loc[~counted, "age_ka_b2k"].min() > 60.202
# Nondecreasing envelopes are required for nonnegative cumulative variance increments.
assert all(np.all(np.diff(g["chronology_envelope_ka"]) >= 0)
           for _, g in grid.groupby("knot_spacing_ka"))

Path("data/processed").mkdir(parents=True, exist_ok=True)
events.to_csv("data/processed/ngrip_warming_cooling_starts.csv", index=False)
grid.to_csv("data/processed/ngrip_chronology_grid.csv", index=False)
events.groupby(["event_type", "chronology_source"]).size().rename("n_events")

event_type  chronology_source
cooling     MCE                  22
            model_ext            13
warming     MCE                  22
            model_ext            12
Name: n_events, dtype: int64